In [ ]:
import os
# --- threshold_test parameters (set by run_all.py through the environment) ---
THRESH_TAG = os.environ.get('THRESH_TAG', 'm2.4')   # m2.4 = baseline cut of -2.4 dex
DATA_DIR = os.path.join('..', '..', 'data', f'thresh_{THRESH_TAG}')
OUT_ROOT = os.path.join('..', '..', 'outputs', f'thresh_{THRESH_TAG}')


In [ ]:
import pandas as pd

In [ ]:
low_mass_satellite_df = pd.read_parquet(os.path.join(DATA_DIR, 'low_mass_satellite_df_annuli_revised.parquet'))

In [ ]:
low_mass_satellite_df

In [ ]:
# # calculating star forming efficiency (SFE) and gas fraction 
# # bin_annuli returns the stellar mass, gas mass, and SFRs in log_10 space
# low_mass_satellite_df['SFE_local'] = 10**(low_mass_satellite_df['avg_SFR'] - low_mass_satellite_df['avg_gas_mass'])
# low_mass_satellite_df['f_gas_local'] = 10**(low_mass_satellite_df['avg_gas_mass'] - low_mass_satellite_df['avg_s_mass'])

# # drop rows where SFR and gas_mass are 0
# low_mass_satellite_df = low_mass_satellite_df[~((low_mass_satellite_df['global_SFRs'] == 0) & (low_mass_satellite_df['gas_mass'] == 0))]

# # creating copy to avoid set with copy warning
# low_mass_satellite_df = low_mass_satellite_df.copy()

# # if the gas mass is 0, set it to default value 10^-3.5
# mask = low_mass_satellite_df['gas_mass'] == 0.0
# low_mass_satellite_df.loc[mask, 'gas_mass'] = 10**(-3.5) * low_mass_satellite_df.loc[mask, 's_mass']


# low_mass_satellite_df['SFE_global'] = low_mass_satellite_df['global_SFRs']/low_mass_satellite_df['gas_mass']
# low_mass_satellite_df['f_gas_global'] = low_mass_satellite_df['gas_mass']/low_mass_satellite_df['s_mass']


In [ ]:
low_mass_satellite_df

In [ ]:
low_mass_satellite_df['quenching_vector'].hist()

In [ ]:
galaxy_class_counts = low_mass_satellite_df.groupby("subID")["quenching_vector"].nunique()

# count galaxies with only one class vs. two classes
num_single_class_galaxies = (galaxy_class_counts == 1).sum()
num_two_class_galaxies = (galaxy_class_counts > 1).sum()

print(f"Number of galaxies with only one class: {num_single_class_galaxies}")
print(f"Number of galaxies with two classes: {num_two_class_galaxies}")

In [ ]:
import numpy as np

# galaxy_class_counts = low_mass_satellite_df.groupby("subID")["quenching_vector"].nunique()


# # single-class and multi-class galaxies
# single_class_galaxies = galaxy_class_counts[galaxy_class_counts == 1].index
# multi_class_galaxies = galaxy_class_counts[galaxy_class_counts > 1].index

# # keep a random fraction
# fraction_to_keep = 1 
# kept_single_class_galaxies = np.random.choice(single_class_galaxies, 
#                                               size=int(len(single_class_galaxies) * fraction_to_keep), 
#                                               replace=False)

# filtered_galaxies = np.concatenate([kept_single_class_galaxies, multi_class_galaxies])

# filtered_df = low_mass_satellite_df[low_mass_satellite_df["subID"].isin(filtered_galaxies)]

# print(f"Original dataset size: {low_mass_satellite_df.shape}")
# print(f"Filtered dataset size: {filtered_df.shape}")


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import RobustScaler

# 'BHMass', 'BHCum_QM', 'Mhalo', and 'overdens'are the features wer are testing
low_mass_satellite_df = low_mass_satellite_df.drop(columns=["class", "BH_cumQM", "BH_cumRM", "global_SFRs", "avg_SFR","avg_gas_mass", "gas_mass", "Bfld", "veldisp", "BH_AGN", "bulge"])
low_mass_satellite_df.duplicated()

# X = low_mass_satellite_df.drop(['quenching_vector'], axis=1)
# y = low_mass_satellite_df['quenching_vector']

In [ ]:
low_mass_satellite_df

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold


# splitting galaxies into test/train based on subID
unique_halos = np.unique(low_mass_satellite_df['GrNr'])

train_aucs = []
test_aucs = []
feature_importances_list = []
feature_names = None
train_f1s = []
test_f1s = []


for seed in range(10):
    train_galaxies, test_galaxies = train_test_split(unique_halos, test_size=0.5, random_state=seed)

    train_data = low_mass_satellite_df[low_mass_satellite_df['GrNr'].isin(train_galaxies)]
    test_data = low_mass_satellite_df[low_mass_satellite_df['GrNr'].isin(test_galaxies)]

    X_train, y_train = train_data.drop(columns=['quenching_vector', 'subID','GrNr']), train_data['quenching_vector']
    X_test, y_test = test_data.drop(columns=['quenching_vector', 'subID','GrNr']), test_data['quenching_vector']

    print(f"Seed {seed}: Train samples: {len(X_train)}, Test samples: {len(X_test)}")
    print(f"Train class distribution: {y_train.value_counts()}")
    print(f"Test class distribution: {y_test.value_counts()}")

    # Add this after your train/test split, before fitting
    print(f"\nSeed {seed}:")
    print(f"  Train quenched fraction: {y_train.mean():.3f}")
    print(f"  Test quenched fraction: {y_test.mean():.3f}")

    # Check how many unique Mhalo values are in train vs test
    print(f"  Train unique galaxies: {len(train_galaxies)}")
    print(f"  Train Mhalo range: {X_train['Mhalo'].min():.1f} - {X_train['Mhalo'].max():.1f}")
    print(f"  Test Mhalo range: {X_test['Mhalo'].min():.1f} - {X_test['Mhalo'].max():.1f}")

    # What fraction of quenched bins come from single-class quenched galaxies
    train_quenched_ids = train_data[train_data['quenching_vector']==1]['subID'].unique()
    test_quenched_ids = test_data[test_data['quenching_vector']==1]['subID'].unique()
    print(f"  Train quenched galaxies: {len(train_quenched_ids)}")
    print(f"  Test quenched galaxies: {len(test_quenched_ids)}")

    if feature_names is None:
        feature_names = X_train.columns.tolist()
    # scaling
    scaler = RobustScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    param_grid = {
    'n_estimators': [500, 600, 700],
    'max_depth': [40, 45, 50],
    'min_samples_split': [100, 125, 150],
    'min_samples_leaf': [500, 525, 550],
    'max_features': [None],
    'bootstrap': [True],
    'class_weight': ['balanced']
    }
    rf = RandomForestClassifier(random_state=seed)
    groups_train = train_data['GrNr']
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)

    random_search = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=param_grid, 
    n_iter=10,
    cv=cv, 
    scoring='roc_auc',
    n_jobs=-1,
    random_state=seed,
    verbose=2)

    random_search.fit(X_train_scaled, y_train, groups=groups_train)

    best_rf = random_search.best_estimator_

    y_train_prob = best_rf.predict_proba(X_train_scaled)[:, 1]
    y_test_prob = best_rf.predict_proba(X_test_scaled)[:, 1]

    fpr_train, tpr_train, _ = roc_curve(y_train, y_train_prob)
    fpr_test, tpr_test, _ = roc_curve(y_test, y_test_prob)

    # ROC curves
    plt.figure(figsize=(8,6))
    plt.plot(fpr_train, tpr_train, label=f'Train AUC = {roc_auc_score(y_train, y_train_prob):.3f}')
    plt.plot(fpr_test, tpr_test, label=f'Test AUC = {roc_auc_score(y_test, y_test_prob):.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend()
    plt.grid(True)
    plt.show()
    print(random_search.best_params_)

    
    # aucs and feature importance
    train_auc = roc_auc_score(y_train, y_train_prob)
    test_auc = roc_auc_score(y_test, y_test_prob)
    gap = train_auc - test_auc
    print(f"Gap (Train - Test AUC): {gap:.4f}")
    print(f"Overfitting ratio (Train/Test AUC): {train_auc / test_auc:.4f}")
    feature_importances_list.append(best_rf.feature_importances_)
    
    train_aucs.append(train_auc)
    test_aucs.append(test_auc)


    y_train_pred = best_rf.predict(X_train_scaled)
    y_test_pred = best_rf.predict(X_test_scaled)

    # f1 scores
    train_f1 = f1_score(y_train, y_train_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    train_f1s.append(train_f1)
    test_f1s.append(test_f1)
    
print("\nAverage AUC over 10 runs:")
print(f"Train AUC: Mean = {np.mean(train_aucs):.4f}, Std = {np.std(train_aucs):.4f}")
print(f"Test AUC:  Mean = {np.mean(test_aucs):.4f}, Std = {np.std(test_aucs):.4f}")
print(f"Train F1: Mean = {np.mean(train_f1s):.4f}, Std = {np.std(train_f1s):.4f}")
print(f"Test F1:  Mean = {np.mean(test_f1s):.4f}, Std = {np.std(test_f1s):.4f}")
gaps = [train - test for train, test in zip(train_aucs, test_aucs)]
overfit_ratios = [train / test for train, test in zip(train_aucs, test_aucs)]
print(f"\nGap stats: Mean = {np.mean(gaps):.4f}, Std = {np.std(gaps):.4f}, Min = {np.min(gaps):.4f}, Max = {np.max(gaps):.4f}")
print(f"Overfitting ratio stats: Mean = {np.mean(overfit_ratios):.4f}, Std = {np.std(overfit_ratios):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
feature_colors = {
    'bin_radii': '#77AADD',
    'avg_s_mass': '#EE8866',
    's_mass': '#FFAABB',
    'BHMass': '#99DDFF',
    'Mhalo': '#44BB99',
    'overdens': '#BBCC33',
    'bulge': '#AAAA00',
    'BH_AGN': '#DDDDDD',
}

# importances + stds across 10 runs
importances_array = np.array(feature_importances_list)
importance_means = np.mean(importances_array, axis=0)
importance_stds = np.std(importances_array, axis=0)

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean Importance': importance_means,
    'Std Dev': importance_stds
}).sort_values(by='Mean Importance', ascending=False)

print("\n=== Average Feature Importances ===")
print(importance_df)

colors = [feature_colors[feat] for feat in importance_df['Feature']]
# Plot
plt.figure(figsize=(10, 6))
plt.bar(importance_df['Feature'], importance_df['Mean Importance'], yerr=importance_df['Std Dev'], capsize=5, color=colors)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Mean Feature Importance")
plt.title("LMS Feature Importances with Error Bars (±1 Std Dev)")
plt.tight_layout()
plt.show()

In [ ]:
low_mass_satellite_df.corr()

In [ ]:
train_mean = round(np.mean(train_aucs), 2)
test_mean = round(np.mean(test_aucs), 2)
print(train_mean)
df = pd.DataFrame({
    'Train AUC': [train_mean],
    'Test AUC': [test_mean]
})

print(df)
import os
# Define your directory and file name
directory = os.path.join(OUT_ROOT, 'rf_classification_outputs')
filename = 'lms_rf_quenching_auc_halosplit.csv'
filepath = os.path.join(directory, filename)

# Create the directory if it doesn't exist
os.makedirs(directory, exist_ok=True)

# Save the DataFrame
df.to_csv(filepath, index=False)

print(f'DataFrame saved to {filepath}')

In [ ]:
import os
# Define your directory and file name
directory = os.path.join(OUT_ROOT, 'rf_classification_outputs')
filename = 'lms_rf_quenching_halosplit.csv'
filepath = os.path.join(directory, filename)

# Create the directory if it doesn't exist
os.makedirs(directory, exist_ok=True)

# Save the DataFrame
importance_df.to_csv(filepath, index=False)

print(f'DataFrame saved to {filepath}')